# Auditoría del modelo — mediciones reproducibles

Este notebook produce **todos los números que el README afirma**. Cada sección corresponde a
una afirmación concreta, y ejecutarlo de principio a fin debe reproducirlas.

Existe porque las conclusiones del proyecto cambiaron tres veces al medirlas, y una
afirmación sobre desempeño que no viene acompañada del código que la produjo no es
verificable por quien lee.

| # | Pregunta | Respuesta obtenida |
|---|---|---|
| 1 | ¿`TIPO_SALTO` filtra el target? | **No** — AUC 0.5104 en solitario |
| 2 | ¿Cuánto del AUC lo produce la partición? | Grupo **0.000**, temporal **−0.101** |
| 3 | ¿Historia completa o ventana reciente? | **1 periodo** gana a 11 años |
| 4 | ¿El stacking aporta? | **No** — pierde contra sus componentes |
| 5 | ¿Sirve recalibrar el umbral cada periodo? | **No** — 85,8 % vs 85,5 % |

Tiempo de ejecución completo: unos 25 minutos.

## Dependencias y datos

In [ ]:
import os
import time
import warnings

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split, GroupShuffleSplit, StratifiedKFold
from sklearn.ensemble import StackingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (roc_auc_score, f1_score, recall_score,
                             precision_score, precision_recall_curve)
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

warnings.filterwarnings('ignore')

Se parte del dataset ya integrado y preprocesado que produce `Prototipo 1.6.ipynb`. Define
`DATASET_AUDITORIA` con su ruta.

> **Limitación conocida.** Ese archivo trae el escalado y la imputación ya aplicados sobre el
> total de los datos, que es precisamente una de las fugas que la versión 1.8 corrige. Por
> eso los experimentos de abajo miden el efecto de la **partición**, del **modelo** y de la
> **ventana**, pero no pueden aislar el efecto del preprocesamiento: eso requiere reejecutar
> desde la base. Se declara aquí para que nadie lea de más en estos resultados.

In [ ]:
RUTA = os.getenv('DATASET_AUDITORIA')
if not RUTA:
    raise RuntimeError("Define DATASET_AUDITORIA con la ruta al dataset preprocesado.")

df = pd.read_excel(RUTA).dropna(subset=['IDENTIFICACION']).reset_index(drop=True)

TARGET = 'TARGET_DESERCION'
IDS = ['IDENTIFICACION', 'PERIODO']
y = df[TARGET].astype(int)
anio = df['AÑO'].astype(int)

print(f"{len(df):,} filas | tasa base {y.mean():.4f}")
print("\nTasa base por año (la deriva que reencuadró el proyecto):")
print(y.groupby(anio).agg(['size', 'mean']).tail(5).round(4))

---
## 1. ¿`TIPO_SALTO` filtra el target?

`TIPO_SALTO` describe la transición al periodo siguiente y el target se deriva de una lógica
emparentada, así que era la sospecha natural de fuga.

Para una sola variable no hace falta entrenar nada. El AUC máximo que puede alcanzar se
obtiene puntuando cada fila con la **tasa empírica de su categoría**: cualquier otra
codificación ordena igual o peor. Ese techo es invariante a cómo esté codificada y responde
la pregunta de forma exacta.

In [ ]:
def auc_por_rangos(score, y):
    """AUC exacto vía Mann-Whitney: (R1 - n1(n1+1)/2) / (n1*n0)."""
    r = pd.Series(score).rank(method='average').to_numpy()
    n1 = int(y.sum()); n0 = len(y) - n1
    return (r[y.to_numpy() == 1].sum() - n1 * (n1 + 1) / 2) / (n1 * n0)


def auc_techo(col, y):
    """AUC máximo alcanzable por una variable sola, invariante a la codificación."""
    return auc_por_rangos(y.groupby(col).transform('mean'), y)


poder = pd.DataFrame(
    [(c, auc_techo(df[c], y), df[c].nunique())
     for c in df.columns if c not in IDS + [TARGET]],
    columns=['variable', 'auc_sola', 'categorias'],
).sort_values('auc_sola', ascending=False).reset_index(drop=True)

poder

**Resultado.** `TIPO_SALTO` obtiene **0.5104**, apenas por encima del azar: no filtra el
target, y la sospecha queda descartada.

Dos observaciones adicionales que salen de la misma tabla:

- **Ninguna variable supera 0.673 por sí sola.** La mejor es `SEMESTRE_SINU`.
- `ESTADO_PAGO` da exactamente 0.5000 con **una sola categoría**: es constante. `ASISTENCIA`
  queda en 0.5010. Ambas se eliminan en la versión 1.8.

In [ ]:
print(f"TIPO_SALTO en solitario : {auc_techo(df['TIPO_SALTO'], y):.4f}")
print(f"Proporción del AUC del modelo completo explicada: "
      f"{(auc_techo(df['TIPO_SALTO'], y) - 0.5) / (0.848 - 0.5) * 100:.1f} %")

print("\nContingencia TIPO_SALTO vs target:")
t = df.groupby('TIPO_SALTO')[TARGET].agg(['count', 'sum', 'mean'])
t.columns = ['n', 'desertores', 'tasa']
print(t.round(4))

---
## 2. ¿Cuánto del desempeño lo produce la partición?

Diseño 2×2: **sujeto** (aleatorio / agrupado por estudiante) × **tiempo** (mezclado /
separado). Mismo modelo, mismos datos, mismas variables. Lo único que cambia es cómo se
separa entrenamiento de prueba, así que cualquier diferencia es atribuible a la partición.

El dataset tiene una fila por estudiante y periodo, con 3,7 filas por alumno en promedio, de
modo que una partición aleatoria deja al mismo estudiante en ambos lados.

In [ ]:
FEATS = [c for c in df.columns if c not in IDS + [TARGET]]
X, g = df[FEATS], df['IDENTIFICACION']


def evaluar(itr, ite, etiqueta):
    ytr, yte = y.iloc[itr], y.iloc[ite]
    m = XGBClassifier(n_estimators=385, max_depth=9, learning_rate=0.0338,
                      subsample=0.6, colsample_bytree=0.6, random_state=42,
                      eval_metric='logloss', n_jobs=-1,
                      scale_pos_weight=(ytr == 0).sum() / max((ytr == 1).sum(), 1))
    t = time.time()
    m.fit(X.iloc[itr], ytr)
    p = m.predict_proba(X.iloc[ite])[:, 1]
    pred = (p >= 0.5).astype(int)
    fila = dict(
        particion=etiqueta,
        auc=round(roc_auc_score(yte, p), 4),
        f1=round(f1_score(yte, pred), 4),
        recall=round(recall_score(yte, pred), 4),
        precision=round(precision_score(yte, pred), 4),
        solape_pct=round(g.iloc[ite].isin(set(g.iloc[itr])).mean() * 100, 1),
        n_test=len(ite),
    )
    print(f"  {etiqueta:<26} AUC {fila['auc']:.4f}  ({time.time()-t:.0f}s)")
    return fila


idx = np.arange(len(df))
resultados = []

# A ─ aleatoria: la que usaba la versión 1.6
a_tr, a_te = train_test_split(idx, test_size=0.3, random_state=42, stratify=y)
resultados.append(evaluar(a_tr, a_te, "A aleatoria"))

# B ─ agrupada por estudiante: ningún alumno en ambos lados
b_tr, b_te = next(GroupShuffleSplit(n_splits=1, test_size=0.3,
                                    random_state=42).split(idx, y, groups=g))
resultados.append(evaluar(b_tr, b_te, "B por estudiante"))

# C ─ temporal: entrena con el pasado, evalúa sobre el futuro
c_tr, c_te = idx[anio <= 2023], idx[anio == 2024]
resultados.append(evaluar(c_tr, c_te, "C temporal"))

# D ─ temporal + grupo: además, solo alumnos nunca vistos
vistos = set(g.iloc[c_tr])
d_te = np.array([i for i in c_te if g.iloc[i] not in vistos])
resultados.append(evaluar(c_tr, d_te, "D temporal + grupo"))

tabla = pd.DataFrame(resultados)
tabla

In [ ]:
base = tabla.loc[0, 'auc']
print(f"AUC con partición aleatoria (réplica de la 1.6): {base:.4f}\n")
for _, r in tabla.iloc[1:].iterrows():
    print(f"  {r['particion']:<22} {r['auc']:.4f}   ({r['auc'] - base:+.4f})")

print(f"\nEfecto de agrupar por sujeto : {tabla.loc[1,'auc'] - base:+.4f}")
print(f"Efecto de separar en el tiempo: {tabla.loc[2,'auc'] - base:+.4f}")

**Resultado.** Descomposición limpia:

- **Sujeto: 0.000.** Eliminar por completo el solape (86,5 % → 0 %) deja el AUC idéntico
  hasta la cuarta cifra. Las variables describen el periodo, no a la persona, así que
  reencontrar al mismo alumno en otro semestre no aporta información sobre su desenlace.
  Contraintuitivo, y por eso valía medirlo.
- **Tiempo: −0.101.** Aquí está el optimismo. La partición aleatoria mezcla periodos.

La fila **D** es la estimación honesta de despliegue: alumnos nuevos, periodo futuro.

Nótese que **D tiene el mejor F1 y la mejor precision de las cuatro**. El modelo ordena peor
sobre el futuro, pero sobre cohortes nuevas —que es donde interviene un programa de
retención— acierta más que lo que sugería el titular.

---
## 3. ¿Historia completa o ventana reciente?

La tasa base sube de forma sostenida entre periodos. Si la población cambia, los datos
antiguos describen otro fenómeno y pueden estorbar más de lo que aportan.

In [ ]:
def entrenar_ventana(mask, etiqueta, modelo=None):
    Xtr, ytr = X[mask], y[mask]
    Xte, yte = X[anio == 2024], y[anio == 2024]
    spw = (ytr == 0).sum() / max((ytr == 1).sum(), 1)
    m = modelo or LGBMClassifier(n_estimators=481, num_leaves=32, learning_rate=0.0244,
                                 subsample=0.9122, random_state=42, n_jobs=-1,
                                 verbose=-1, scale_pos_weight=spw)
    t = time.time()
    m.fit(Xtr, ytr)
    p = m.predict_proba(Xte)[:, 1]
    fila = dict(ventana=etiqueta, n_train=len(ytr), tasa=round(ytr.mean(), 4),
                auc=round(roc_auc_score(yte, p), 4), segundos=round(time.time() - t))
    print(f"  {etiqueta:<30} n={fila['n_train']:>7,}  AUC={fila['auc']:.4f}  "
          f"({fila['segundos']}s)")
    return fila, p


ventanas = [
    (anio <= 2023,                     "historia completa"),
    ((anio >= 2021) & (anio <= 2023),  "ventana 3 periodos"),
    ((anio >= 2022) & (anio <= 2023),  "ventana 2 periodos"),
    (anio == 2023,                     "ventana 1 periodo"),
]
filas, probas = [], {}
for m, et in ventanas:
    f, p = entrenar_ventana(m, et)
    filas.append(f); probas[et] = p

pd.DataFrame(filas)

**Resultado.** Un periodo le gana a once años de historia, con el 13,6 % de los datos y siete
veces menos tiempo de entrenamiento. Es la firma de la deriva de concepto: acumular historia
no solo no ayuda, contamina.

---
## 4. ¿El stacking aporta algo?

El ensamblado XGBoost + LightGBM fue la sofisticación central de la versión 1.6, resultado de
una optimización bayesiana. Se compara contra sus propios componentes por separado, sobre la
ventana ganadora.

In [ ]:
Xtr_a, ytr_a = X[anio == 2023], y[anio == 2023]
Xte_a, yte_a = X[anio == 2024], y[anio == 2024]
spw = (ytr_a == 0).sum() / (ytr_a == 1).sum()
verdad = yte_a.to_numpy()


def xgb():
    return XGBClassifier(n_estimators=385, max_depth=9, learning_rate=0.0338,
                         subsample=0.6, colsample_bytree=0.6, random_state=42,
                         eval_metric='logloss', n_jobs=-1, scale_pos_weight=spw)


def lgbm():
    return LGBMClassifier(n_estimators=481, num_leaves=32, learning_rate=0.0244,
                          subsample=0.9122, random_state=42, n_jobs=-1,
                          verbose=-1, scale_pos_weight=spw)


candidatos = {
    'XGBoost solo': xgb(),
    'LightGBM solo': lgbm(),
    'Stacking XGB+LGBM': StackingClassifier(
        estimators=[('XGB', xgb()), ('LGBM', lgbm())],
        final_estimator=LogisticRegression(max_iter=1000, random_state=42),
        cv=StratifiedKFold(n_splits=3), n_jobs=1, passthrough=True),
}

filas = []
for nombre, m in candidatos.items():
    t = time.time()
    m.fit(Xtr_a, ytr_a)
    p = m.predict_proba(Xte_a)[:, 1]
    orden = np.argsort(-p)
    fila = {'arquitectura': nombre, 'auc': round(roc_auc_score(yte_a, p), 4),
            'segundos': round(time.time() - t)}
    for pct in (1, 5, 10):
        k = int(len(verdad) * pct / 100)
        fila[f'P@{pct}%'] = round(float(verdad[orden[:k]].mean()), 4)
    filas.append(fila)
    print(f"  {nombre:<20} AUC {fila['auc']:.4f}  ({fila['segundos']}s)")

pd.DataFrame(filas)

**Resultado.** El stacking es **el peor de los tres** bajo validación temporal, peor que
cualquiera de sus dos componentes, y nueve veces más lento. LightGBM solo gana en todas las
métricas.

Bajo partición aleatoria el stacking sí mejoraba. Esa ganancia era sobre una partición que
sobrestimaba, y no sobrevive a una evaluación honesta.

---
## 5. ¿Sirve recalibrar el umbral en cada periodo?

Si el umbral óptimo deriva entre periodos, recalibrarlo contra datos etiquetados recientes
debería recuperar desempeño. Se comparan cuatro formas de elegirlo, todas evaluadas sobre el
mismo periodo futuro.

In [ ]:
def umbral_optimo(y_true, proba):
    p, r, thr = precision_recall_curve(y_true, proba)
    f1 = 2 * p * r / (p + r + 1e-9)
    return float(thr[int(np.argmax(f1[:-1]))])


m = lgbm()
m.fit(X[anio <= 2022], y[anio <= 2022])
p_ca = m.predict_proba(X[anio == 2023])[:, 1]
p_te = m.predict_proba(X[anio == 2024])[:, 1]
p_tr = m.predict_proba(X[anio <= 2022])[:, 1]

umbrales = {
    'fijo 0.5': 0.5,
    'calibrado en entrenamiento': umbral_optimo(y[anio <= 2022], p_tr),
    'calibrado en periodo anterior': umbral_optimo(y[anio == 2023], p_ca),
    'calibrado en el propio test (oráculo)': umbral_optimo(y[anio == 2024], p_te),
}

yte_f = y[anio == 2024]
filas = [{'estrategia': k, 'umbral': round(u, 4),
          'f1': round(f1_score(yte_f, (p_te >= u).astype(int)), 4),
          'recall': round(recall_score(yte_f, (p_te >= u).astype(int)), 4),
          'precision': round(precision_score(yte_f, (p_te >= u).astype(int)), 4)}
         for k, u in umbrales.items()]
pd.DataFrame(filas)

**Resultado.** Recalibrar contra el periodo anterior recupera **85,8 %** del máximo
alcanzable; no recalibrar en absoluto recupera **85,5 %**. La diferencia es ruido.

El motivo está en los umbrales: el óptimo del periodo anterior es casi idéntico al del
entrenamiento, mientras el del periodo objetivo se desploma. **La deriva no es suave, salta**,
así que el pasado inmediato no predice el corte del futuro.

De aquí sale la decisión de diseño central de la versión 1.8: si el umbral no es transferible
pero el ranking sí —el AUC no depende de dónde se calibre—, el sistema debe operar por
**capacidad de atención** y no por probabilidad de corte.

---
## Resumen

| Afirmación del README | Sección | Estado |
|---|---|---|
| `TIPO_SALTO` no filtra el target (0.5104) | 1 | descartada por medición |
| Fuga por grupo = 0.000 | 2 | descartada por medición |
| Fuga temporal = −0.101 | 2 | **confirmada** |
| Estimación honesta AUC 0.764 | 2 | confirmada |
| 1 periodo > 11 años de historia | 3 | confirmada |
| El stacking pierde contra sus componentes | 4 | confirmada |
| Recalibrar el umbral no aporta | 5 | descartada por medición |

Tres de las siete hipótesis que motivaron esta auditoría resultaron falsas. Quedan
documentadas junto a las confirmadas: descartar una causa con una medición vale tanto como
confirmarla, y ahorra que alguien vuelva a perseguirla.